# The transformer encoder, from the CNN side: code behind the numbers

Companion to the AI 4200 lecture *The transformer encoder, from the CNN side*. It reproduces every
computed number on the slides: the 4 × 4 running example (recap, max and softmax pooling, attention,
two heads), the ÷√d experiment, the position experiments (CNN vs attention with no position, learned
position, offset bias, RoPE), the stalled-bias diagnosis and the number-of-heads experiment.

**Running it.** Everything runs on a CPU or a Colab T4. The small worked examples take seconds. The
training experiments (sections for slides 15, 19–22 and 24) train about 60 small models; on a CPU that is
a few hours, on a T4 much less. Set `QUICK = True` for a two-minute smoke test (tiny data, one epoch,
one seed). Results are cached to disk after every run, so an interrupted run resumes.

The slides were computed on a CPU; a GPU gives slightly different floating-point results, so a training
accuracy can differ in the last digit. The ImageNet table and the picture-word result are not computed
here (see their sections).

In [ ]:
QUICK = False            # True: a few steps on tiny data, one seed (smoke test only)
USE_DRIVE_CACHE = False  # on Colab: keep the results cache in Google Drive

import json
import math
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

BLUE, ORANGE, INK, GREY = "#2557D6", "#C2510F", "#16181D", "#8E94A0"

CACHE_DIR = "cache"
if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = "/content/drive/MyDrive/transformer-encoder-cache"
if QUICK:
    CACHE_DIR = CACHE_DIR + "-quick"   # never mix smoke-test results with real ones
os.makedirs(CACHE_DIR, exist_ok=True)

## Shared helpers

`softmax_rows` is used by all the worked examples. The rest of the helpers (data, models, training)
are defined further down, where the experiments start.

In [ ]:
def softmax_rows(scores):
    """Softmax along the last axis, shifted by the row maximum for numerical stability."""
    exps = np.exp(scores - scores.max(axis=-1, keepdims=True))
    return exps / exps.sum(axis=-1, keepdims=True)

## Slide 2 · Recap: CNNs and the running example

The recap animation runs one 4 × 4 image through a 3 × 3 sharpening filter, ReLU, 2 × 2 max pooling,
global average pooling and a linear layer. We pick the feature map first (it is the running example of
the whole lecture) and solve for the image that produces it: a convolution is linear in its input, so
it is a 16 × 16 matrix we can invert. The linear layer's weights are illustrative, not trained.

In [ ]:
feature_map = np.array([
    [0.2, 0.9, 0.1, 0.4],
    [0.5, 0.3, 0.8, 0.6],
    [0.7, 0.1, 0.3, 0.2],
    [0.4, 0.6, 0.9, 0.5],
])
kernel = np.array([
    [0.0, -0.25, 0.0],
    [-0.25, 2.0, -0.25],
    [0.0, -0.25, 0.0],
])


def conv_same(image, k):
    """3x3 cross-correlation with zero padding 1 (what nn.Conv2d computes)."""
    padded = np.pad(image, 1)
    out = np.zeros_like(image)
    for r in range(image.shape[0]):
        for c in range(image.shape[1]):
            out[r, c] = (padded[r:r + 3, c:c + 3] * k).sum()
    return out


# column i of A is the convolution of the i-th unit image
A = np.zeros((16, 16))
for i in range(16):
    unit = np.zeros(16)
    unit[i] = 1
    A[:, i] = conv_same(unit.reshape(4, 4), kernel).flatten()
image = np.linalg.solve(A, feature_map.flatten()).reshape(4, 4)
assert np.allclose(conv_same(image, kernel), feature_map)

quadrants = {"TL": feature_map[:2, :2], "TR": feature_map[:2, 2:], "BL": feature_map[2:, :2], "BR": feature_map[2:, 2:]}
max_pool = {name: q.max() for name, q in quadrants.items()}
global_avg = np.mean(list(max_pool.values()))

W_linear = np.array([[2.0], [-1.0], [0.5]])   # illustrative weights: 1 channel in, 3 class scores out
b_linear = np.array([-0.5, 0.4, 0.0])
y_hat = np.array([global_avg]) @ W_linear.T + b_linear

print("image (rounded):\n", image.round(2))
print("max pool:", max_pool)
print("global average:", global_avg)
print("class scores y_hat:", y_hat.round(3))

Slide: image entries 0.22 … 0.35, max pool 0.9 / 0.8 / 0.7 / 0.9, average 0.825, scores +1.15, −0.42,
+0.41 (the second score is −0.425, which the slide rounds to −0.42).

## Slides 7–9 · Max pooling is a gated sum; hardmax → softmax pooling

Max pooling multiplies each value in a window by a gate (1 for the winner, 0 for the rest) and sums.
Softening the hardmax into `softmax(x / τ)` with τ = 0.1 gives every value a small gate, so every value
gets a gradient.

In [ ]:
TAU = 0.1
for name, q in quadrants.items():
    x = q.flatten()
    hard_gates = (x == x.max()).astype(float)
    soft_gates = softmax_rows(x / TAU)
    print(f"{name}: x = {x}, hardmax gates {hard_gates} -> {hard_gates @ x:.1f}, "
          f"softmax gates {soft_gates.round(3)} -> {soft_gates @ x:.3f}")

Slide 9: softmax gates 0.001 / 0.979 / 0.018 / 0.002 for TL and outputs 0.891, 0.770, 0.663, 0.891.

## Slides 10–13 · Softmax pooling → attention; every quadrant queries every quadrant; context blended in

Each quadrant is now a feature vector (blue, wings, hull). Picture A has a bird in the top-right
quadrant, picture B a boat. The projections are hand-picked: the query reads only *blue* ("is there an
object nearby?"), the keys answer with *wings* and *hull*, and W_V = I. Scores here are not divided by
√d (that comes on slide 15).

In [ ]:
QUAD_NAMES = ["TL", "TR", "BL", "BR"]
pictures = {
    "A (bird)": np.array([[1, 0, 0], [1, 1, 0], [1, 0, 0], [1, 0, 0]], dtype=float),
    "B (boat)": np.array([[1, 0, 0], [1, 0, 1], [1, 0, 0], [1, 0, 0]], dtype=float),
}
W_Q = np.array([[2, 0, 0], [2, 0, 0]], dtype=float)   # blue asks: is there an object nearby?
W_K = np.array([[0, 1, 0], [0, 0, 1]], dtype=float)   # wings and hull answer
W_V = np.eye(3)                                       # hand over the features as they are


def attention(X, W_Q, W_K, W_V, scale=False):
    """Single-head attention over the rows of X. Returns gates and output."""
    Q, K, V = X @ W_Q.T, X @ W_K.T, X @ W_V.T
    scores = Q @ K.T
    if scale:
        scores = scores / np.sqrt(Q.shape[1])
    gates = softmax_rows(scores)
    return Q, K, scores, gates, gates @ V


for name, X in pictures.items():
    Q, K, scores, gates, out = attention(X, W_Q, W_K, W_V)
    print(f"Picture {name}")
    print("  queries:", Q.tolist(), " keys:", K.tolist())
    print("  scores of TL:", scores[0], " gates:", gates[0].round(3))
    print("  attention output (every quadrant):", out.round(2).tolist())
    print("  x + attention (residual):", (X + out).round(2).tolist())

Slides 10–11: q = [2, 2] for every quadrant, scores [0, 2, 0, 0], gates 0.096 / 0.711 / 0.096 / 0.096,
output [1, 0.71, 0]. Slides 12–13: with the residual, top-left becomes [2, 0, 0.71] in picture B and
[2, 0.71, 0] in picture A.

## Slide 16 · Two heads, two questions

Two heads with d = 1 each. Head 1 looks for wings and hands over the wings channel; head 2 looks for a
hull and hands over the hull channel. Concatenating the two maps and multiplying by W_O = [+1, −1]
gives one map: positive for sky, negative for water.

In [ ]:
heads = {
    "head 1 (wings nearby?)": dict(W_Q=np.array([[2.0, 0, 0]]), W_K=np.array([[0, 1.0, 0]]), W_V=np.array([[0, 1.0, 0]])),
    "head 2 (hull nearby?)": dict(W_Q=np.array([[2.0, 0, 0]]), W_K=np.array([[0, 0, 1.0]]), W_V=np.array([[0, 0, 1.0]])),
}
W_O = np.array([[1.0], [-1.0]])

for name, X in pictures.items():
    head_maps = [attention(X, **h)[-1] for h in heads.values()]   # each (4 quadrants, 1 channel)
    concat = np.concatenate(head_maps, axis=1)                    # 4 x 2
    merged = concat @ W_O                                         # 4 x 1
    print(f"Picture {name}: concat (rows TL, TR, BL, BR)\n{concat.round(2)}\n  x W_O = {merged.ravel().round(2)}")

Slide: 0.71 / 0.00 for picture A gives +0.71 everywhere (sky); picture B gives −0.71 (water).

## Slide 15 · Variance through attention (the ÷√d part)

(a) With unit-variance q and k, the score q · k has variance d, so as d grows the softmax over 64 keys
approaches a hardmax. Dividing by √d keeps the largest gate small. (b) The same effect in training: the
toy model of the next sections with one head of width 256, with and without the scale. Part (b) needs the
helpers defined in the next section, so it runs there.

In [ ]:
generator = torch.Generator().manual_seed(0)
TOKENS, QUERIES = 64, 500
dims = [1, 4, 16, 64, 256]
top_gate_unscaled, top_gate_scaled = [], []
for d in dims:
    q = torch.randn(QUERIES, d, generator=generator)
    k = torch.randn(TOKENS, d, generator=generator)
    scores = q @ k.T
    top_gate_unscaled.append(scores.softmax(-1).max(-1).values.mean().item())
    top_gate_scaled.append((scores / math.sqrt(d)).softmax(-1).max(-1).values.mean().item())
    print(f"d = {d:3d}: score std {scores.std().item():6.2f}, largest gate {top_gate_unscaled[-1]:.2f}, "
          f"with ÷√d {top_gate_scaled[-1]:.2f}")

plt.figure(figsize=(6, 3.5))
plt.plot(dims, top_gate_unscaled, "o-", color=ORANGE, label="QKᵀ")
plt.plot(dims, top_gate_scaled, "o-", color=BLUE, label="QKᵀ / √d")
plt.xscale("log", base=2)
plt.xlabel("head width d")
plt.ylabel("largest of 64 gates (mean)")
plt.title("Without ÷√d the softmax turns into a hardmax")
plt.legend()
plt.show()

Slide: largest gate 0.14, 0.29, 0.56, 0.79, 0.90; with ÷√d 0.14, 0.11, 0.10, 0.11, 0.11.

## Helpers for the experiments: data, models, training

**Data.** 40 × 40 black images with a square and a circle; the label is "is the square left of the
circle?". Shapes stay 6 px from the border (so zero padding gives no absolute position) and their
centres are at least 12 px apart along x (so no small patch sees both). The heads experiment adds a
triangle and a second question, "is the triangle above the circle?", for 4 classes.

**Models.** A CNN with global average pooling at the end; the same first two conv layers with the
pooling straight after them; and a toy hybrid ViT: two 3 × 3 convs, a 4 × 4 patch conv (a 10 × 10 grid
of patches), pre-norm encoder blocks with multi-head attention, then average over patches and a
linear layer. Position can be given as a learned absolute embedding, a learned offset bias
b[Δrow, Δcol] added to every score, or 2D RoPE.

**Training.** Adam (lr 2 × 10⁻³) with a cosine schedule, batch 128. `train` can give a parameter group
its own learning-rate multiplier and can call a logger every step (used for the stalled-bias slide).

In [ ]:
IMG = 40     # image side in pixels
MARGIN = 6   # shapes stay this far from the border
GAP = 12     # minimum centre distance along the axis that decides the label


def draw_square(img, x, y, size=6):
    img[y:y + size, x:x + size] = 1.0


def draw_circle(img, cx, cy, r=3.5):
    yy, xx = np.mgrid[0:IMG, 0:IMG]
    img[(xx - cx) ** 2 + (yy - cy) ** 2 <= r ** 2] = 1.0


def draw_triangle(img, x, y, size=7):
    """Upward triangle with its apex at the top centre of a size-by-size box."""
    for row in range(size):
        half = row // 2
        mid = x + size // 2
        img[y + row, mid - half:mid + half + 1] = 1.0


def boxes_apart(a, b, gap=2):
    """True if two (x, y, size) boxes do not overlap (with a small gap)."""
    ax, ay, asz = a
    bx, by, bsz = b
    return ax + asz + gap <= bx or bx + bsz + gap <= ax or ay + asz + gap <= by or by + bsz + gap <= ay


def place(rng, size):
    """Random top-left corner for a shape of this size, MARGIN px from every border."""
    x = int(rng.integers(MARGIN, IMG - MARGIN - size + 1))
    y = int(rng.integers(MARGIN, IMG - MARGIN - size + 1))
    return (x, y, size)

In [ ]:
def make_left_of(n, seed):
    """Square and circle anywhere; label 1 if the square is left of the circle."""
    rng = np.random.default_rng(seed)
    images = np.zeros((n, 1, IMG, IMG), dtype=np.float32)
    labels = np.zeros(n, dtype=np.int64)
    for i in range(n):
        while True:
            square = place(rng, 6)
            circle = place(rng, 8)
            if boxes_apart(square, circle, gap=4) and abs((square[0] + 3) - (circle[0] + 4)) >= GAP:
                break
        draw_square(images[i, 0], square[0], square[1])
        draw_circle(images[i, 0], circle[0] + 3.5, circle[1] + 3.5)
        labels[i] = int(square[0] + 3 < circle[0] + 4)
    return torch.tensor(images), torch.tensor(labels)


def make_two_relations(n, seed):
    """Square, circle and triangle. 4 classes = 2 * (square left of circle) + (triangle above circle)."""
    rng = np.random.default_rng(seed)
    images = np.zeros((n, 1, IMG, IMG), dtype=np.float32)
    labels = np.zeros(n, dtype=np.int64)
    for i in range(n):
        while True:
            square = place(rng, 6)
            circle = place(rng, 8)
            triangle = place(rng, 7)
            apart = (boxes_apart(square, circle, gap=4) and boxes_apart(square, triangle, gap=4)
                     and boxes_apart(circle, triangle, gap=4))
            clear_x = abs((square[0] + 3) - (circle[0] + 4)) >= GAP
            clear_y = abs((triangle[1] + 3.5) - (circle[1] + 4)) >= GAP
            if apart and clear_x and clear_y:
                break
        draw_square(images[i, 0], square[0], square[1])
        draw_circle(images[i, 0], circle[0] + 3.5, circle[1] + 3.5)
        draw_triangle(images[i, 0], triangle[0], triangle[1])
        left = int(square[0] + 3 < circle[0] + 4)
        above = int(triangle[1] + 3.5 < circle[1] + 4)
        labels[i] = 2 * left + above
    return torch.tensor(images), torch.tensor(labels)

In [ ]:
class CNN(nn.Module):
    """Traditional CNN: global average pooling only at the very end."""

    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 20 x 20
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 10 x 10
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 5 x 5
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(),
        )
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))


class EarlyPoolCNN(nn.Module):
    """Same first two conv layers, but global average pooling straight after them."""

    def __init__(self, n_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
        )
        self.head = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, n_classes))

    def forward(self, x):
        return self.head(self.features(x).mean(dim=(2, 3)))

In [ ]:
def rope_2d(t, rows, cols):
    """Rotate pairs of channels by patch position: first half by column, second half by row.

    t: (batch, heads, tokens, head_dim); rows, cols: (tokens,) integer positions.
    """
    half = t.shape[-1] // 2
    rotated_parts = []
    for part, pos in ((t[..., :half], cols), (t[..., half:], rows)):
        n_pairs = half // 2
        freqs = 1.0 / (10.0 ** (torch.arange(n_pairs, device=t.device) / n_pairs))
        angle = pos[:, None].float() * freqs[None, :]          # (tokens, pairs)
        cos, sin = angle.cos(), angle.sin()
        a, b = part[..., 0::2], part[..., 1::2]
        rotated = torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)
        rotated_parts.append(rotated)
    return torch.cat(rotated_parts, dim=-1)


class Attention(nn.Module):
    """Multi-head self-attention; each head has its own query, key and value projections."""

    def __init__(self, d_model, n_heads, head_dim, use_rope, scale=True, offset_bias=False, grid=10):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.use_rope = use_rope
        self.scale = scale
        self.grid = grid
        # offset bias: one learned number per (row offset, column offset), per head, added to the score
        self.bias = nn.Parameter(torch.zeros(n_heads, 2 * grid - 1, 2 * grid - 1)) if offset_bias else None
        self.qkv = nn.Linear(d_model, 3 * n_heads * head_dim)
        self.out = nn.Linear(n_heads * head_dim, d_model)

    def forward(self, x, rows, cols):
        batch, tokens, _ = x.shape
        q, k, v = self.qkv(x).view(batch, tokens, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        if self.use_rope:
            q = rope_2d(q, rows, cols)
            k = rope_2d(k, rows, cols)
        scores = q @ k.transpose(-2, -1)
        if self.scale:
            scores = scores / math.sqrt(self.head_dim)
        if self.bias is not None:
            d_row = rows[:, None] - rows[None, :] + self.grid - 1
            d_col = cols[:, None] - cols[None, :] + self.grid - 1
            scores = scores + self.bias[:, d_row, d_col]
        mixed = scores.softmax(dim=-1) @ v                       # (batch, heads, tokens, head_dim)
        return self.out(mixed.transpose(1, 2).reshape(batch, tokens, -1))


class EncoderBlock(nn.Module):
    """Pre-norm encoder block: attention and a feed-forward net, each with a residual add."""

    def __init__(self, d_model, n_heads, head_dim, use_rope, scale=True, offset_bias=False, grid=10):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.attn = Attention(d_model, n_heads, head_dim, use_rope, scale, offset_bias, grid)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, 2 * d_model), nn.ReLU(), nn.Linear(2 * d_model, d_model))

    def forward(self, x, rows, cols):
        x = x + self.attn(self.norm1(x), rows, cols)
        return x + self.ffn(self.norm2(x))


class Hybrid(nn.Module):
    """Toy hybrid ViT: conv stem, 4x4 patch conv, encoder blocks, average over patches, linear."""

    def __init__(self, n_classes=2, use_rope=False, n_blocks=2, n_heads=4, head_dim=16,
                 d_model=64, patch=4, scale=True, learned_pos=False, offset_bias=False):
        super().__init__()
        self.front = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
        )
        self.patchify = nn.Conv2d(32, d_model, patch, stride=patch)
        grid = IMG // patch
        rows, cols = torch.meshgrid(torch.arange(grid), torch.arange(grid), indexing="ij")
        self.register_buffer("rows", rows.flatten())
        self.register_buffer("cols", cols.flatten())
        # original-ViT style: one learned vector per patch position, added to the patch embedding
        self.pos = nn.Parameter(torch.randn(1, grid * grid, d_model) * 0.02) if learned_pos else None
        self.blocks = nn.ModuleList(
            [EncoderBlock(d_model, n_heads, head_dim, use_rope, scale, offset_bias, grid) for _ in range(n_blocks)])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, x):
        x = self.patchify(self.front(x)).flatten(2).transpose(1, 2)   # (batch, patches, d_model)
        if self.pos is not None:
            x = x + self.pos
        for block in self.blocks:
            x = block(x, self.rows, self.cols)
        return self.head(self.norm(x).mean(dim=1))

In [ ]:
def evaluate(model, images, labels, batch=500):
    """Test accuracy, computed in batches."""
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch):
            xb, yb = images[i:i + batch].to(device), labels[i:i + batch].to(device)
            correct += (model(xb).argmax(1) == yb).sum().item()
    return correct / len(images)


def train(model, train_set, test_set, epochs, lr=2e-3, batch=128, seed=0, lr_mult=None, logger=None):
    """Adam with a cosine schedule. Returns (test accuracy, per-step losses).

    lr_mult: {parameter-name suffix: multiplier}, e.g. {"attn.bias": 10} gives the offset bias 10x the lr.
    logger: optional object with before_step(model, optimizer) and after_step(model, optimizer).
    """
    torch.manual_seed(seed)
    model.to(device)
    images, labels = train_set
    lr_mult = lr_mult or {}
    groups = {}
    for name, param in model.named_parameters():
        mult = next((m for suffix, m in lr_mult.items() if name.endswith(suffix)), 1)
        groups.setdefault(mult, []).append(param)
    optimizer = torch.optim.Adam([{"params": ps, "lr": lr * m} for m, ps in groups.items()], lr=lr)
    steps = epochs * math.ceil(len(images) / batch)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, steps)
    losses = []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(images))
        for i in range(0, len(images), batch):
            idx = order[i:i + batch]
            xb, yb = images[idx].to(device), labels[idx].to(device)
            loss = F.cross_entropy(model(xb), yb)
            optimizer.zero_grad()
            loss.backward()
            if logger is not None:
                logger.before_step(model, optimizer)
            optimizer.step()
            schedule.step()
            if logger is not None:
                logger.after_step(model, optimizer)
            losses.append(loss.item())
        if logger is not None:
            logger.end_epoch(model, test_set)
    return evaluate(model, *test_set), losses


def cached(name, key, run):
    """Run `run()` once per key and keep the result in a JSON cache, so an interrupted sweep resumes."""
    path = os.path.join(CACHE_DIR, name + ".json")
    cache = json.load(open(path)) if os.path.exists(path) else {}
    if key not in cache:
        start = time.time()
        cache[key] = run()
        json.dump(cache, open(path, "w"))
        print(f"  ran {key} in {time.time() - start:.0f} s")
    return cache[key]

In [ ]:
# The datasets every experiment uses (seeds fixed so the slides' numbers come back)
N_TRAIN, N_TEST = (400, 200) if QUICK else (8000, 2000)
EPOCH_SCALE = 0.1 if QUICK else 1      # QUICK: 1 epoch instead of 10
SEEDS = [0] if QUICK else [0, 1, 2]


def epochs(n):
    """Number of epochs for a run that the slides trained for n epochs."""
    return max(1, round(n * EPOCH_SCALE))


left_of_train = make_left_of(N_TRAIN, seed=100)
left_of_test = make_left_of(N_TEST, seed=200)

fig, axes = plt.subplots(1, 6, figsize=(10, 2))
for ax, image, label in zip(axes, left_of_train[0][:6], left_of_train[1][:6]):
    ax.imshow(image[0], cmap="gray")
    ax.set_title("left of" if label == 1 else "not left of", fontsize=9)
    ax.axis("off")
fig.suptitle("Square left of the circle? (first six training images)")
plt.show()

### Slide 15 (part b) · ÷√d in training

The hybrid with RoPE, two blocks and one head of width 256, trained for 6 epochs with and without the
scale. Without it the scores are so large that the gates saturate and the model stays at chance.

In [ ]:
sqrtd_runs = {}
for scale in (True, False):
    def run_sqrtd(scale=scale):
        torch.manual_seed(0)
        model = Hybrid(use_rope=True, n_blocks=2, n_heads=1, head_dim=256, scale=scale)
        acc, losses = train(model, left_of_train, left_of_test, epochs(6), seed=0)
        return [acc, losses]
    sqrtd_runs[scale] = cached("sqrtd", str(scale), run_sqrtd)
    print(f"scale = {scale}: test accuracy {sqrtd_runs[scale][0]:.3f}")

plt.figure(figsize=(7, 3.5))
plt.plot(sqrtd_runs[True][1], color=BLUE, label="QKᵀ / √d")
plt.plot(sqrtd_runs[False][1], color=ORANGE, label="QKᵀ")
plt.xlabel("training step")
plt.ylabel("cross-entropy loss")
plt.title("Head width 256: training with and without ÷√d")
plt.legend()
plt.show()

Slide: accuracy 1.00 with ÷√d, 0.51 without (computed: 1.000 and 0.508).

## Slides 19–21 · Experiment: relative position; absolute vs relative position; offset bias

Every model is trained on the same 8,000 images for 10 epochs with seeds 0, 1 and 2. Attention with
no position cannot tell left from right; a learned absolute position works on some seeds; a learned
offset bias needs a larger learning rate on its table (slide 22 shows why); RoPE works out of the box.
The 30-epoch rows are the controls mentioned at the bottom of slide 21.

In [ ]:
# name: (build the model, epochs, extra training settings)
position_models = {
    "CNN, pooling at the end": (lambda: CNN(), 10, {}),
    "CNN, pooling early": (lambda: EarlyPoolCNN(), 10, {}),
    "Hybrid, no position": (lambda: Hybrid(use_rope=False), 10, {}),
    "Hybrid + RoPE": (lambda: Hybrid(use_rope=True), 10, {}),
    "Hybrid + learned position": (lambda: Hybrid(learned_pos=True), 10, {}),
    "Hybrid + offset bias": (lambda: Hybrid(offset_bias=True), 10, {}),
    "Hybrid + offset bias, 10x bias lr": (lambda: Hybrid(offset_bias=True), 10, {"lr_mult": {"attn.bias": 10}}),
    "Hybrid + offset bias, 30 epochs": (lambda: Hybrid(offset_bias=True), 30, {}),
    "Hybrid, no position, 30 epochs": (lambda: Hybrid(use_rope=False), 30, {}),
    "Hybrid + learned position, 30 epochs": (lambda: Hybrid(learned_pos=True), 30, {}),
    "Hybrid + RoPE, 30 epochs": (lambda: Hybrid(use_rope=True), 30, {}),
}
if QUICK:   # smoke test: one model of each kind
    position_models = {k: v for k, v in position_models.items() if "30 epochs" not in k}

position_results = {}
for name, (build, n_epochs, extra) in position_models.items():
    accs = []
    for seed in SEEDS:
        def run_position(build=build, n_epochs=n_epochs, extra=extra, seed=seed):
            torch.manual_seed(seed)
            return train(build(), left_of_train, left_of_test, epochs(n_epochs), seed=seed, **extra)[0]
        accs.append(cached("position", f"{name}|{seed}", run_position))
    position_results[name] = accs
    print(f"{name:38s} " + " · ".join(f"{a:.2f}" for a in accs) + f"   mean {np.mean(accs):.2f}")

Slides 19–21: CNN pooling at the end 1.00, pooling early 0.51, no position 0.58 · 0.57 · 0.59
(mean 0.58), learned position 0.81 · 0.51 · 1.00 (mean 0.77), offset bias 0.59 · 0.58 · 0.58,
offset bias with 10× lr on its table 1.00 on every seed, RoPE 1.00. At 30 epochs every model with
position reaches 1.00; no position stays at 0.59.

## Slide 22 · A stalled parameter: how to spot it

Why does the offset bias fail at the default learning rate? Adam moves each parameter by about lr per
step whatever the gradient, so in 630 steps of a cosine schedule an entry can move at most
Σ lr_t ≈ 0.63. To single out one patch of 100, a bias needs a score gap of about ln 100 ≈ 4.6. We retrain
seed 0 of both offset-bias runs and log, every step, the largest |b| in each block's table, the
"reach" Σ lr_t, and the gradient norm.

In [ ]:
class BiasLogger:
    """Logs the offset-bias tables every step: largest |b|, reach (sum of lr) and gradient norm."""

    def __init__(self):
        self.reach = 0.0
        self.log = {"reach": [], "max_abs_b": [], "grad_norm": [], "epoch_test_acc": []}

    def bias_tables(self, model):
        return [p for n, p in model.named_parameters() if n.endswith("attn.bias")]

    def before_step(self, model, optimizer):
        self.log["grad_norm"].append(max(b.grad.norm().item() for b in self.bias_tables(model)))
        self.reach += optimizer.param_groups[-1]["lr"]   # the bias tables are in the last group

    def after_step(self, model, optimizer):
        self.log["reach"].append(self.reach)
        self.log["max_abs_b"].append(max(b.detach().abs().max().item() for b in self.bias_tables(model)))

    def end_epoch(self, model, test_set):
        self.log["epoch_test_acc"].append(evaluate(model, *test_set))


bias_logs = {}
for run_name, mult in {"default lr": 1, "10x bias lr": 10}.items():
    def run_diagnosis(mult=mult):
        torch.manual_seed(0)
        model = Hybrid(offset_bias=True)
        logger = BiasLogger()
        lr_mult = {"attn.bias": mult} if mult != 1 else None
        train(model, left_of_train, left_of_test, epochs(10), seed=0, lr_mult=lr_mult, logger=logger)
        return logger.log
    bias_logs[run_name] = cached("stalled_bias", run_name, run_diagnosis)
    log = bias_logs[run_name]
    print(f"{run_name:12s}: largest |b| {log['max_abs_b'][-1]:.2f}, reach {log['reach'][-1]:.2f}, "
          f"gradient norm {min(log['grad_norm']):.1e} to {max(log['grad_norm']):.1e}, "
          f"test accuracy {log['epoch_test_acc'][-1]:.2f}")
print(f"needed score gap: ln 100 = {math.log(100):.1f}")

In [ ]:
plt.figure(figsize=(7, 4))
for (run_name, log), color in zip(bias_logs.items(), (ORANGE, BLUE)):
    plt.plot(log["max_abs_b"], color=color, label=f"largest |bias| entry, {run_name}")
    plt.plot(log["reach"], color=color, linestyle="--", label=f"reach Σ lr, {run_name}")
plt.axhline(math.log(100), color=GREY, linestyle=":", label="needed ≈ ln 100")
plt.xlabel("training step")
plt.ylabel("size")
plt.title("The offset bias can only grow as far as Adam's reach")
plt.legend(fontsize=8)
plt.show()

Slide: default lr reach 0.63, largest entry 0.49; 10× lr reach 6.3, largest 5.6. The slide gives the
gradient norm as 10⁻⁶ to 10⁻³; the logged range is about 3 × 10⁻⁷ to 2 × 10⁻³ (the smallest values come
in the last steps, when the schedule's lr is almost zero).

## Slide 24 · Experiment: number of heads

Two relations at once (square left of circle? triangle above circle?), 4 classes. The hybrid has RoPE,
one encoder block, and heads of width 4, so more heads also means more width. Seeds 0 and 1.

In [ ]:
two_rel_train = make_two_relations(N_TRAIN, seed=300)
two_rel_test = make_two_relations(N_TEST, seed=400)

heads_results = {}
for n_heads in [1, 2, 4, 8]:
    accs = []
    for seed in SEEDS[:2]:
        def run_heads(n_heads=n_heads, seed=seed):
            torch.manual_seed(seed)
            model = Hybrid(n_classes=4, use_rope=True, n_blocks=1, n_heads=n_heads, head_dim=4, d_model=32)
            return train(model, two_rel_train, two_rel_test, epochs(10), seed=seed)[0]
        accs.append(cached("heads", f"{n_heads}|{seed}", run_heads))
    heads_results[n_heads] = accs
    print(f"{n_heads} heads: " + " / ".join(f"{a:.2f}" for a in accs) + f"   mean {np.mean(accs):.2f}")

plt.figure(figsize=(5, 3.5))
plt.bar([str(h) for h in heads_results], [np.mean(a) for a in heads_results.values()], color=BLUE)
plt.axhline(0.25, color=GREY, linestyle=":", label="chance")
plt.xlabel("number of heads")
plt.ylabel("test accuracy (mean of seeds)")
plt.title("Two relations need two heads")
plt.legend()
plt.show()

Slide: 1 head 0.52 (seeds 0.43 / 0.61), 2, 4 and 8 heads 1.00.

## Slide 29 · Embeddings

The embedding table on this slide is illustrative (3 numbers per word, not trained). The point it makes
is one line of linear algebra: a one-hot row times the table picks out one row.

In [ ]:
vocab = ["the", "cat", "sat", "mat"]
E = np.array([[0.1, 0.0, 0.3], [0.9, 0.4, -0.2], [-0.3, 0.8, 0.5], [0.7, 0.5, -0.1]])   # illustrative
one_hot_cat = np.array([0, 1, 0, 0])
print("one-hot('cat') @ E =", one_hot_cat @ E, "= row of 'cat':", E[vocab.index("cat")])

## Slide 31 · ImageNet accuracy by pretraining data

Not computed here: the numbers are from the papers. ViT: Dosovitskiy et al., 2021, Appendix Table 5;
CNN: BiT ResNet152x4, Kolesnikov et al., 2020, Tables 1–2.

## Slide 33 · What the picture adds

Not computed here: the instructor's own visual masked-language-model run (a marimo notebook, trained on
captioned photos), not part of this notebook.